# Mosaic TIFF files from 4 folders

This notebook:
1. Takes 4 input folders containing `.tif` / `.tiff` files
2. Collects all raster files from those folders
3. Mosaics them into a single output raster
4. Saves the mosaic to the chosen output path


In [ ]:
# Install dependency if needed
# Uncomment this in Jupyter if rasterio is not already installed:
# !pip install rasterio


In [1]:
from pathlib import Path
import subprocess

def get_tiff_files(folder_path):
    folder = Path(folder_path)
    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder}")
    return sorted(list(folder.glob("*.tif")) + list(folder.glob("*.tiff")))

def mosaic_tiffs_vrt(folders, output_tif, vrt_path=None):
    if not isinstance(folders, (list, tuple)) or len(folders) == 0:
        raise ValueError("Provide at least one folder.")

    all_tiffs = []
    for i, folder in enumerate(folders, start=1):
        tif_files = get_tiff_files(folder)
        print(f"Folder {i}: {folder}")
        print(f"  TIFF files found: {len(tif_files)}")
        all_tiffs.extend([str(f) for f in tif_files])

    if not all_tiffs:
        raise ValueError("No TIFF files found.")

    output_tif = Path(output_tif)
    output_tif.parent.mkdir(parents=True, exist_ok=True)

    if vrt_path is None:
        vrt_path = output_tif.with_suffix(".vrt")
    else:
        vrt_path = Path(vrt_path)

    # Save tif list to a text file
    tif_list_file = output_tif.parent / "tiff_list.txt"
    with open(tif_list_file, "w", encoding="utf-8") as f:
        for tif in all_tiffs:
            f.write(tif + "\n")

    # Build VRT
    subprocess.run(
        ["gdalbuildvrt", "-input_file_list", str(tif_list_file), str(vrt_path)],
        check=True
    )

    # Convert VRT to compressed tiled GeoTIFF
    result = subprocess.run(
    [
        "gdal_translate",
        str(vrt_path),
        str(output_tif),
        "-co", "TILED=YES",
        "-co", "COMPRESS=LZW",
        "-co", "BIGTIFF=YES"
    ],
    capture_output=True,
    text=True
    )

    print("Return code:", result.returncode)
    print("STDOUT:\n", result.stdout)
    print("STDERR:\n", result.stderr)

    print(f"Saved mosaic to: {output_tif}")
    return output_tif

## Example usage

Replace the folder paths below with your own paths.

In [2]:
# Example paths
folder1 = [r"D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\planet_downloads_PSScene_rgb_2025\orders_all"]
folder2 = [r"D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\planet_downloads_PSScene_rgb_2025\orders_all_delta"]
folder3 = [r"D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\planet_downloads_PSScene_rgb_2025\orders_all_delta2"]
folder4 = [r"D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\planet_downloads_PSScene_rgb_2025\orders_all_delta3"]
folders = [r"D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\planet_downloads_PSScene_rgb_2025\orders_all",r"D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\planet_downloads_PSScene_rgb_2025\orders_all_delta",r"D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\planet_downloads_PSScene_rgb_2025\orders_all_delta2",r"D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\planet_downloads_PSScene_rgb_2025\orders_all_delta3"]
output_path = r"D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\mosaic.tif"

# Run
mosaic_tiffs_vrt(folders, output_path)


Folder 1: D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\planet_downloads_PSScene_rgb_2025\orders_all
  TIFF files found: 71
Folder 2: D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\planet_downloads_PSScene_rgb_2025\orders_all_delta
  TIFF files found: 50
Folder 3: D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\planet_downloads_PSScene_rgb_2025\orders_all_delta2
  TIFF files found: 80
Folder 4: D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\planet_downloads_PSScene_rgb_2025\orders_all_delta3
  TIFF files found: 108
Return code: 0
STDOUT:
 Input file size is 207575, 133247
0...10...20...30...40...50...60...70...80...90...100 - done in 01:00:32.

STDERR:
 
Saved mosaic to: D:\Thesis\glacial_lake_detection_thesis\Inference\2 Getting imageries from planet\mosaic.tif


WindowsPath('D:/Thesis/glacial_lake_detection_thesis/Inference/2 Getting imageries from planet/mosaic.tif')

## Notes

- All TIFFs should have the same CRS and compatible resolution for best results.
- If they differ, `rasterio.merge` will still try to merge them, but output may need checking.
- If your TIFF files are inside subfolders too, I can give you a version that searches recursively.
